# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()

# Print dataset name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.


In [ ]:
# List all record sets with their @id and name
print("Record Sets available in the dataset:")
for record_set in dataset.metadata.record_sets:
    print(f"  @id: {record_set.id}, name: {record_set.name}")

# Let's detail the fields/columns of each record set
print("\nFields/Columns for each Record Set:")
for record_set in dataset.metadata.record_sets:
    print(f"\nRecord Set: {record_set.name} (@id: {record_set.id})")
    print("Fields and their @id's:")
    for field in getattr(record_set, 'fields', []):
        print(f"  - {field.name} (@id: {field.id}, dataType: {field.data_type})")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Identify all record set @id's
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for {record_set_id} with shape {dataframes[record_set_id].shape}")
    else:
        print(f"No records found for record set {record_set_id}")

# For demonstration, let's show columns of the first non-empty record set:
main_record_set_id = None
for rsid, df in dataframes.items():
    if len(df.columns) > 0:
        main_record_set_id = rsid
        break

if main_record_set_id:
    print(f"\nColumns in record set {main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes with columns found.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA: Find a numeric field and group/categorize records
import numpy as np

# Pick numeric field and group field (if any known numeric columns, otherwise try generic numeric pattern)
numeric_field = None
group_field = None

if main_record_set_id:
    df = dataframes[main_record_set_id]
    numeric_candidates = [c for c in df.columns if df[c].dtype in [np.float64, np.int64] or df[c].apply(lambda x: isinstance(x, (int, float))).all()]
    if len(numeric_candidates) == 0:
        # Try to coerce numeric columns
        for col in df.columns:
            coerced = pd.to_numeric(df[col], errors='coerce')
            if coerced.notnull().sum() > 0:
                df[col] = coerced
                numeric_candidates.append(col)
        df = df.copy()
    if len(numeric_candidates) > 0:
        numeric_field = numeric_candidates[0]

    # For group field, pick first non-numeric
    group_candidates = [c for c in df.columns if c != numeric_field and (df[c].dtype == object or df[c].dtype.name == 'category')]
    if len(group_candidates) > 0:
        group_field = group_candidates[0]

    print(f"Selected numeric_field: {numeric_field}")
    print(f"Selected group_field: {group_field}")

    # Filter for values above mean
    threshold = df[numeric_field].mean() if numeric_field else 0
    filtered_df = df[df[numeric_field] > threshold] if numeric_field else df
    print(f"\nFiltered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    if numeric_field:
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group and aggregate
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field}:")
        display(grouped_df.head())
else:
    print("No usable record set found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for the numeric field
if main_record_set_id and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[main_record_set_id][numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If group_field, show boxplot
    if group_field:
        plt.figure(figsize=(12, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=dataframes[main_record_set_id])
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No suitable numeric field for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated loading and exploring the [FAIR⁲] mass spectrometry analysis dataset using the `mlcroissant` library.
- Record sets, their fields (referenced by `@id`), and data were programmatically discovered and extracted.
- Example filtering, normalization, and grouping by record set fields enabled quick exploratory data analysis.
- Data visualizations offer insights into value distributions and relationships between key attributes.
- Next steps: further domain-driven analyses, exploration by biological significance, and use as input for scientific/ML workflows.
